## Import and utilitaries

In [64]:
# file name for the reward table
file_name = "../try_trafficlight/reward_df_50agents_sharpthreshold_smallestamplitude_2.csv"

In [ ]:
import pandas as pd
import numpy as np

df_reward = pd.read_csv(file_name)
tab_reward = np.zeros((10,1024))
for i in range(10):
    tab_reward[i] = df_reward[str(i)].values

tab_reward[0]

array([-0.7, -0.7, -0.7, ..., -1.2, -1.2, -1.2], shape=(1024,))

In [3]:
def id_to_strategy(id):
    strategy = [0 for _ in range(10)]
    i = 0
    while id > 0:
        if id % 2 == 1:
            strategy[9-i] = 1
        id = id//2
        i += 1
    return strategy


def strategy_to_id(s):
    id = 0
    for i in range(10):
        id += s[i]*(2**(9-i))
    return id

In [ ]:
def reward(i,id):
    return float(tab_reward[i][id])

def id_to_reward(id):
    return [reward(i,id) for i in range(10)]

def s_to_reward(s):
    id = strategy_to_id(s)
    return id_to_reward(id)

def neighbouring_strategies(s):
    neigh = [ [s[j] for j in range(10)] for _ in range(10)]
    for i in range(10):
        neigh[i][i] = 1 - neigh[i][i]
    return neigh

def neighbouring_ids(id):
    s = id_to_strategy(id)
    t = neighbouring_strategies(s)
    neigh = []
    for i in range(10):
        neigh.append(strategy_to_id(t[i]))
    return neigh

In [6]:
def partitions(ensemble):
    if len(ensemble) == 0:
        yield []
        return

    premier = ensemble[0]
    for partition in partitions(ensemble[1:]):
        # Insérer le premier élément dans chaque sous-ensemble de la partition
        for i in range(len(partition)):
            yield partition[:i] + [[premier] + partition[i]] + partition[i+1:]
        # Créer un nouveau sous-ensemble pour le premier élément
        yield [[premier]] + partition

# Exemple d'utilisation pour l'ensemble [0, 1, 2, ..., 9]
ensemble = list(range(10))
partitions_list = list(partitions(ensemble))

# Afficher le nombre total de partitions et quelques exemples
print(f"Nombre total de partitions : {len(partitions_list)}")
print("Quelques exemples de partitions :")
for p in partitions_list[:49]:  # Afficher seulement les 5 premières partitions
    print(p)

Nombre total de partitions : 115975
Quelques exemples de partitions :
[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]
[[0], [1, 2, 3, 4, 5, 6, 7, 8, 9]]
[[0, 1], [2, 3, 4, 5, 6, 7, 8, 9]]
[[1], [0, 2, 3, 4, 5, 6, 7, 8, 9]]
[[0], [1], [2, 3, 4, 5, 6, 7, 8, 9]]
[[0, 1, 2], [3, 4, 5, 6, 7, 8, 9]]
[[1, 2], [0, 3, 4, 5, 6, 7, 8, 9]]
[[0], [1, 2], [3, 4, 5, 6, 7, 8, 9]]
[[0, 2], [1, 3, 4, 5, 6, 7, 8, 9]]
[[2], [0, 1, 3, 4, 5, 6, 7, 8, 9]]
[[0], [2], [1, 3, 4, 5, 6, 7, 8, 9]]
[[0, 1], [2], [3, 4, 5, 6, 7, 8, 9]]
[[1], [0, 2], [3, 4, 5, 6, 7, 8, 9]]
[[1], [2], [0, 3, 4, 5, 6, 7, 8, 9]]
[[0], [1], [2], [3, 4, 5, 6, 7, 8, 9]]
[[0, 1, 2, 3], [4, 5, 6, 7, 8, 9]]
[[1, 2, 3], [0, 4, 5, 6, 7, 8, 9]]
[[0], [1, 2, 3], [4, 5, 6, 7, 8, 9]]
[[0, 2, 3], [1, 4, 5, 6, 7, 8, 9]]
[[2, 3], [0, 1, 4, 5, 6, 7, 8, 9]]
[[0], [2, 3], [1, 4, 5, 6, 7, 8, 9]]
[[0, 1], [2, 3], [4, 5, 6, 7, 8, 9]]
[[1], [0, 2, 3], [4, 5, 6, 7, 8, 9]]
[[1], [2, 3], [0, 4, 5, 6, 7, 8, 9]]
[[0], [1], [2, 3], [4, 5, 6, 7, 8, 9]]
[[0, 1, 3], [2, 4, 5, 6, 7,

## Finding pure Nash equilibria

In [12]:
def nash_deviation(s):
    rew = s_to_reward(s)
    neigh = neighbouring_strategies(s)
    dev = []
    for i in range(10):
        alternative = s_to_reward(neigh[i])[i]
        dev.append(alternative - rew[i])
    return dev

def nash_equilibrium(s):
    dev = nash_deviation(s)
    for i in range(1,10):
        # we do not verify if the first AV can deviate. Otherwise, no non-trivial NE will be found.
        if dev[i] > 0:
            return False
    return True

def nash_equilibrium_eps(s,eps=0):
    dev = nash_deviation(s)
    for i in range(1,10):
        # we do not verify if the first AV can deviate. Otherwise, no non-trivial NE will be found.
        if dev[i] > eps:
            return False
    return True

In [13]:
for id in range(1024):
    if nash_equilibrium(id_to_strategy(id)):
        print(id_to_strategy(id))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [14]:
nash_deviation(id_to_strategy(0))

[-0.43333333333333335,
 -1.2666666666666662,
 -1.1833333333333331,
 -1.15,
 -0.7666666666666668,
 -0.75,
 -0.6333333333333333,
 -0.5666666666666669,
 -0.5,
 -0.40000000000000013]

In [15]:
P = [[2,3,5],[0,1,9,8],[4],[6,7]]

## Useful functions for coalitions

In [16]:
def id_to_reward_coalition(id,coalition):
    sum = 0
    nb = 0
    for i in coalition:
        sum += reward(i,id)
        nb += 1
    if nb == 0:
        return -1e5
    return sum/nb

def s_to_reward_coalition(s,coalition):
    id = strategy_to_id(s)
    return id_to_reward_coalition(id,coalition)

def id_to_reward_partition(id,partition):
    return [id_to_reward_coalition(id,coalition) for coalition in partition]

def s_to_reward_partition(s,partition):
    id = strategy_to_id(s)
    return id_to_reward_partition(id,partition)

def tab_reward_partition(partition):
    tab = []
    for id in range(1024):
        tab.append(id_to_reward_partition(id,partition))
    return [[tab[id][c] for id in range(1024)] for c in range(len(partition))]

In [17]:
def compatible_strategies(coalition,x):
    '''
    coalition is a table of k indexes from [n]
    x is a table of k bits

    output is the table of strategies where forall i in [k], s[coalition[i]] == x[i]
    '''
    k = len(coalition)
    comp = []
    id = 0
    while id < 1024:
        s = id_to_strategy(id)
        i = 0
        while i<k and (s[coalition[i]] == x[i]):
            i += 1
        if i == k:
            comp.append(s)
        id += 1
    return comp

def compatible_ids(coalition,id):
    s = id_to_strategy(id)
    x = [s[i] for i in coalition]
    comp_s = compatible_strategies(coalition,x)
    return [strategy_to_id(t) for t in comp_s]

In [18]:
def coalition_to_strategy(coalition):
    s = [0 for _ in range(10)]
    for i in coalition:
        s[i] = 1
    return s

def strategy_to_coalition(s):
    coalition = []
    for i in range(10):
        if s[i] == 1:
            coalition.append(i)
    return coalition

def reward_coalition_strategy(coalition,x):
    s_comp = compatible_strategies(coalition,x)
    rew = 0
    for s in s_comp:
        g = s_to_reward_coalition(s,coalition)
        if g < rew:
            rew = g
    return rew

In [19]:
def neighbouring_coalitions(coalition):
    s = coalition_to_strategy(coalition)
    neigh_s = neighbouring_strategies(s)
    return [strategy_to_coalition(neigh_s[i]) for i in range(10)]

## Coalition worth : Min-max paradigm

The worth of a coalition is defined as the value guaranteed by the coalition, in a two-player game (coalition vs the rest) using the reward table.

In [ ]:
def worth_coalition(coalition):
    k = len(coalition)
    rew = -1e5
    best_x = [0 for _ in range(k)]
    for id in range(2**k):
        x = [(id//(2**i))%2 for i in range(k)]
        v = reward_coalition_strategy(coalition,x)
        if v > rew:
            rew = v
            best_x = x
    return rew

tab_worth = [-1e-5]
for id in range(1,1024):
    c = strategy_to_coalition(id_to_strategy(id))
    tab_worth.append(worth_coalition(c))
tab_worth

In [ ]:
def search_worth(coalition):
    return tab_worth[strategy_to_id(coalition_to_strategy(coalition))]

def worth_coalition_partition(partition):
    return [search_worth(coalition) for coalition in partition]

### Coalition and partition stability (using min-max paradigm)

In [ ]:
def internal_stability(coalition):
    v = search_worth(coalition)
    for i in coalition:
        if worth_coalition([i]) > v:
            print("not internally stable,", i, "quits")
            return False
    return True

def stability(coalition):
    s = coalition_to_strategy(coalition)
    v = search_worth(coalition)
    for i in range(10):
        if s[i] == 1:
            s[i] = 0
            cprime = strategy_to_coalition(s)
            if len(cprime) > 0:
                if search_worth(cprime) > v:
                    print("not internally stable,", i, "is kicked")
                    return False
                if search_worth([i]) > v:
                    print("not internally stable,", i, "quits")
                    return False
            s[i] = 1
        if s[i] == 0:
            s[i] = 1
            cprime = strategy_to_coalition(s)
            if search_worth(cprime) > v and search_worth(cprime) > search_worth([i]):
                print("not externally stable,", i, "joins")
                return False
            s[i] = 0
    return True

def partition_stability(partition):
    s_list = [coalition_to_strategy(coalition) for coalition in partition]
    v_list = worth_coalition_partition(partition)
    for j in range(len(s_list)):
        c = partition[j]
        s = s_list[j]
        v = v_list[j]
        for i in c:
            s[i] = 0
            cprime = strategy_to_coalition(s)
            if len(cprime) > 0:
                if search_worth(cprime) > v:
                    # print("not internally stable,", i, "is kicked")
                    return False
                if search_worth([i]) > v:
                    # print("not internally stable,", i, "quits")
                    return False
            s[i] = 1
        for k in range(len(s_list)):
            ck = partition[k]
            sk = s_list[k]
            vk = v_list[k]
            for i in ck:
                s[i] = 1
                cprime = strategy_to_coalition(s)
                if search_worth(cprime) > v and search_worth(cprime) > vk:
                    # print("not externally stable,", i, "moves to", j)
                    return False
                s[i] = 0
    return True


In [ ]:
def partition_stability_eps(partition):
    cost_kick, cost_isol, cost_comm, cost_move, cost_vote = 0,0,0,0,0
    s_list = [coalition_to_strategy(coalition) for coalition in partition]
    v_list = worth_coalition_partition(partition)
    for j in range(len(s_list)):
        c = partition[j]
        s = s_list[j]
        v = v_list[j]
        for i in c:
            s[i] = 0
            cprime = strategy_to_coalition(s)
            if len(cprime) > 0:
                if search_worth(cprime) - v > cost_kick:
                    cost_kick = search_worth(cprime) - v
                if search_worth([i]) - v > cost_isol:
                    cost_isol = search_worth([i]) - v
                if search_worth(cprime) - v > cost_vote*len(cprime):
                    cost_vote = (search_worth(cprime) - v)/len(cprime)
            s[i] = 1
        for k in range(len(s_list)):
            ck = partition[k]
            sk = s_list[k]
            vk = v_list[k]
            for i in ck:
                s[i] = 1
                cprime = strategy_to_coalition(s)
                if search_worth(cprime) - v > cost_comm and search_worth(cprime)-vk > cost_move:
                    # print("not externally stable,", i, "moves to", j)
                    if search_worth(cprime) - v > cost_comm:
                        cost_comm = search_worth(cprime) - v
                    if search_worth(cprime)-vk > cost_move:
                        cost_move = search_worth(cprime)-vk
                s[i] = 0
    return cost_kick, cost_isol, cost_comm, cost_move,cost_vote

In [ ]:
stable = []
for id in range(1,1024):
    print(id)
    coalition = strategy_to_coalition(id_to_strategy(id))
    if stability(coalition):
        stable.append(id)
stable

In [ ]:
[strategy_to_coalition(id_to_strategy(id)) for id in stable]

In [ ]:
partition_stability([[0],[1,2],[3],[4],[5],[6],[7,8],[9]])

In [ ]:
stab_p = []
i=0
for p in partitions_list:
    i += 1
    if partition_stability(p):
        stab_p.append(p)
    if i%100 == 0:
        print(i, "faits,", len(stab_p), "partitions stables trouvées")
print(len(stab_p))

### Computing costs for all sources of instability

In [ ]:
i=0
kick, isol, comm, move, vote = [], [], [], [], []
'''
These calculate cost thresholds, beyond which a partition is stable because the gains
from an action are not worth its cost.
It is the maximum "potential cliff" descending to a neighboring state.
For example:

kick is the cost threshold beyond which no coalition kicks one of its members
isol is the cost threshold beyond which no agent isolates itself from its coalition
comm is the cost threshold beyond which no coalition integrates a new member
move is the cost threshold beyond which no agent switches coalitions.
vote is the cost threshold beyond which no coalition kicks one of its members,
    when this cost is divided among the remaining members of the coalition.
    (aka no agent votes to kick a fellow member from its coalition)

!!! Warning
comm and move are not independent and would better be described by a Pareto set for each partition.
'''
for p in partitions_list:
    i += 1
    cost_kick, cost_isol, cost_comm, cost_move,cost_vote = partition_stability_eps(p)
    kick.append(cost_kick)
    isol.append(cost_isol)
    comm.append(cost_comm)
    move.append(cost_move)
    vote.append(cost_vote)
    if i%10000 == 0:
        print(i, "done")

In [ ]:
stab_p

In [ ]:
P = stab_p[0]
P

In [ ]:
worth_coalition_partition(P)

In [ ]:
partition_stability([[i] for i in range(10)])

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(np.array(move),np.array(comm),alpha=0.05)
plt.show()

## Agent reward : Onur gammas

There is only one active coalition.
Agent reward (when the agent is part of the coalition) is a linear combination of the individual travel time and the coalition travel time.

In [ ]:
gamma_indiv = 0
gamma_coal = 1

def onur(indiv,coal):
    return gamma_indiv*indiv+gamma_coal*coal

In [ ]:
def id_to_reward_onur(id,coalition):
    coalition_reward = id_to_reward_coalition(id,coalition)
    rew = []
    for i in range(10):
        rew.append(reward(i,id))
    for i in coalition:
        rew[i] = onur(rew[i],coalition_reward)
    return rew

def s_to_reward_onur(s,coalition):
    id = strategy_to_id(s)
    return id_to_reward_onur(id,coalition)

In [ ]:
tab_reward_onur = [[] for id in range(1024)]
for id_s in range(1024):
    s = id_to_strategy(id_s)
    for id_c in range(1024):
        coalition = strategy_to_coalition(id_to_strategy(id_c))
        tab_reward_onur[id_s].append(s_to_reward_onur(s,coalition))
    
tab_reward_onur

In [ ]:
def nash_deviation_onur(s,coalition):
    neigh_s = neighbouring_strategies(s)
    neigh_c = neighbouring_coalitions(coalition)
    dev = []
    id_s = strategy_to_id(s)
    id_c = strategy_to_id(coalition_to_strategy(coalition))
    rew = tab_reward_onur[id_s][id_c]
    for i in range(10):
        id_ns = strategy_to_id(neigh_s[i])
        id_nc = strategy_to_id(coalition_to_strategy(neigh_c[i]))
        change_s = tab_reward_onur[id_ns][id_c][i]
        change_c = tab_reward_onur[id_s][id_nc][i]
        change_both = tab_reward_onur[id_ns][id_nc][i]
        # print(max(change_s,change_c,change_both) - rew[i],"\t",rew[i],change_s,change_c,change_both)
        dev.append(max(change_s,change_c,change_both)-rew[i])
    return dev

def nash_equilibrium_onur(s,coalition):
    dev = nash_deviation_onur(s,coalition)
    for i in range(1,10):
        # we do not verify if the first AV can deviate. Otherwise, no non-trivial NE will be found.
        if dev[i] > 0:
            return False
    return True

In [ ]:
nash_equilibrium_onur(id_to_strategy(12),[0,3,5])

In [ ]:
for id_s in range(1024):
    s = id_to_strategy(id_s)
    for id_c in range(1024):
        c = strategy_to_coalition(id_to_strategy(id_c))
        if nash_equilibrium_onur(s,c):
            print(s,c)

In [ ]:
gamma_indiv, gamma_coal = 0.5, 0.5
for id_s in range(1024):
    s = id_to_strategy(id_s)
    for id_c in range(1024):
        c = strategy_to_coalition(id_to_strategy(id_c))
        if nash_equilibrium_onur(s,c):
            print(s,c)

## Computing correlated equilibrium for coalitions.

Coalitions share information. For each opponent strategy, there exists a correlated equilibrium minimizing average travel time for the coalition. This solution may be better than any pure strategy the coalition comes up with.

In [ ]:
import cvxpy as cp


In [ ]:
Nmax = 10
N = list(range(Nmax))
S = list(range(2**Nmax))
R = np.array(tab_reward)
a = np.array([id_to_strategy(id) for id in S]) # Bitmask matrix a[s][i] = (s // 2**i) % 2 ---
x = cp.Variable(len(S))  # Vector of size 2**Nmax
total_time = cp.sum(cp.multiply(x, R.sum(axis=0)))  # sum_s x[s] * sum_i R[s,i]
R

In [ ]:
def build_fixed_route(s,coalition):
    f0 = []
    f1 = []
    cs = coalition_to_strategy(coalition)
    for i in N:
        if cs[i] == 0:
            if s[i] == 0:
                f0.append(i)
            else:
                f1.append(i)
    return f0,f1

def build_constraints(coalition,f0,f1):
    cons = []

    # Normalization
    cons.append(cp.sum(x) == 1)

    # No deviation zero
    for i in coalition:
        lhs = cp.sum([x[s] * R[i,s] for s in S if a[s, i] == 0])
        rhs = cp.sum([x[s] * R[i,s + 2**i] for s in S if a[s, i] == 0 and (s + 2**i) in S])
        cons.append(lhs >= rhs)

    # No deviation one
    for i in coalition:
        lhs = cp.sum([x[s] * R[i,s] for s in S if a[s, i] == 1])
        rhs = cp.sum([x[s] * R[i,s - 2**i] for s in S if a[s, i] == 1 and (s - 2**i) in S])
        cons.append(lhs >= rhs)

    for s in S:
        # Fixed to route 0: if a[s,i] = 1 → x[s] = 0
        for i in f0:
            if a[s, i] == 1:
                cons.append(x[s] == 0)
        # Fixed to route 1: if a[s,i] = 0 → x[s] = 0
        for i in f1:
            if a[s, i] == 0:
                cons.append(x[s] == 0)

    # Bounds
    cons.append(x >= 0)
    cons.append(x <= 1)

    return cons

def correlated_equilibrium(s,coalition):
    FIXED_ROUTE_0, FIXED_ROUTE_1 = build_fixed_route(s, coalition)
    constraints = build_constraints(coalition,FIXED_ROUTE_0, FIXED_ROUTE_1)

    prob = cp.Problem(cp.Maximize(total_time), constraints)
    prob.solve()
    
    return x.value

In [ ]:
ce = correlated_equilibrium(id_to_strategy(478),[1,8,9])
print("total_time =",total_time.value)
for id in S:
    if ce[id] > 1e-5:
        print(ce[id], id, id_to_strategy(id))

In [ ]:
for id in S:
    if x.value[id] > 1e-5:
        print(x.value[id], id)

## Agent reward : Cobb-Douglas

In [ ]:
alpha = 0

def cobbdouglas(indiv,coal):
    return indiv**(alpha)*coal**(1-alpha)

## Study table

In [ ]:
tab_reward[:,2]
id_to_strategy(2)

In [ ]:
def travel_times(id):
    s = id_to_strategy(id)
    t0 = 0
    t1 = 0
    n0 = 0
    n1 = 0
    les_t = tab_reward[:,id]
    for i in range(10):
        if s[i] == 0:
            n0 += 1
            t0 += les_t[i]
        else:
            n1 += 1
            t1 += les_t[i]
    if n0 > 0:
        t0 = t0/n0
    if n1 > 0:
        t1 = t1/n1
    return n0, n1, t0, t1

In [ ]:
travel_times(2)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

import matplotlib.lines as lines

cmap = mpl.cm.cool
norm = mpl.colors.Normalize(vmin=5, vmax=10)

In [ ]:
viridis = colormaps["viridis"]
viridis(0.5)

In [ ]:
les_t0 = [[] for n0 in range(11)]
les_t1 = [[] for n0 in range(11)]

fig = plt.figure()
ax = fig.add_subplot()

for id in range(1024):
    n0,n1,t0,t1 = travel_times(id)
    les_t0[n0].append(t0)
    les_t1[n0].append(t1)
for n0 in range(1,10):
    ax.scatter(les_t0[n0],les_t1[n0],label="n0 = %s, n1 = %s"%(n0,10-n0),alpha = 0.5,color = viridis(n0/10))

ax.set_xlabel("t0")
ax.set_ylabel("t1")

ax.add_artist(lines.Line2D([-2,0], [-2,0], linewidth=2))

fig.legend()
fig.show()

## Is the initial situation a Strong NE?

In [43]:
def strong_nash_equilibrium(s):
    rew = s_to_reward(s)
    for id in range(1,1024):
        coalition = strategy_to_coalition(id_to_strategy(id))
        neigh_s = [s[i] for i in range(10)]
        for i in coalition:
            neigh_s[i] = 1 - s[i]
        alt = s_to_reward(neigh_s)
        flag = True
        for i in coalition:
            if alt[i] - rew[i] < 0:
                flag = False
        if flag:
            print(coalition)
            print(rew)
            print(alt)
    return True

def strong_nash_equilibrium_TU(s):
    rew = s_to_reward(s)
    for id in range(1,1024):
        coalition = strategy_to_coalition(id_to_strategy(id))
        neigh_s = [s[i] for i in range(10)]
        for i in coalition:
            neigh_s[i] = 1 - s[i]
        alt = s_to_reward(neigh_s)

        gain = 0

        for i in coalition:
            gain += alt[i] - rew[i]
        if gain > 0:
            print(coalition)
            print(rew)
            print(alt)
    return True

In [66]:
strong_nash_equilibrium(id_to_strategy(0))

[7]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3833333333333333, -2.4166666666666665, -2.35, -2.5166666666666666]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.8666666666666667, -2.4, -2.3666666666666667, -2.5]
[6]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3833333333333333, -2.4166666666666665, -2.35, -2.5166666666666666]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3666666666666667, -2.3833333333333333, -2.8833333333333333, -2.4833333333333334]
[2]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3833333333333333, -2.4166666666666665, -2.35, -2.5166666666666666]
[-0.7, -0.7333333333333333, -1.2666666666666666, -1.4666666666666666, -1.65, -1.65, -2.35, -2.4, -2.333333333333333, -2.5]


True

In [67]:
strong_nash_equilibrium_TU(id_to_strategy(0))

[7]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3833333333333333, -2.4166666666666665, -2.35, -2.5166666666666666]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.8666666666666667, -2.4, -2.3666666666666667, -2.5]
[6]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3833333333333333, -2.4166666666666665, -2.35, -2.5166666666666666]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3666666666666667, -2.3833333333333333, -2.8833333333333333, -2.4833333333333334]
[2]
[-0.7, -0.7333333333333333, -1.3833333333333333, -1.5, -1.6666666666666667, -1.6833333333333331, -2.3833333333333333, -2.4166666666666665, -2.35, -2.5166666666666666]
[-0.7, -0.7333333333333333, -1.2666666666666666, -1.4666666666666666, -1.65, -1.65, -2.35, -2.4, -2.333333333333333, -2.5]
[1, 2, 3, 4, 5, 6, 7, 8]
[-0.7, 

True

In [ ]:
id_to_reward(0), id_to_reward(3)